# cusmic demo

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rndsrc/cusmic/blob/main/demo/demo.ipynb)

Select **Runtime -> Change runtime type -> GPU**, then **Run all**.
This notebook recreates the [L.A.Cosmic user-guide image](https://lacosmic.readthedocs.io/en/stable/user_guide/index.html)
and compares cusmic with the CPU reference.


In [ ]:
%pip install -q "cupy-cuda12x[ctk]"

%pip install -q matplotlib
%pip install -q lacosmic

# Use the installed CuPy wheel instead of building CuPy from source.
%pip install -q --no-deps git+https://github.com/rndsrc/cusmic.git

## Reference example

The guide uses a 512 x 512 image, 200 cosmic-ray trails, and seed 0.
`baseline` includes noise before cosmic rays are added.


In [ ]:
import cupy as cp
import cusmic
import lacosmic
import numpy as np
from lacosmic.utils import make_cosmic_rays, make_gaussian_sources
from matplotlib import pyplot as plt
from matplotlib.colors import Normalize, PowerNorm

assert cp.cuda.runtime.getDeviceCount() > 0, "Select a GPU runtime."

baseline, error = make_gaussian_sources((512, 512), seed=0)
cosmics  = make_cosmic_rays(baseline.shape, n_cosmics=200, seed=0)
image    = baseline + cosmics

In [ ]:
settings = {
    'contrast'          :1, 
    'cr_threshold'      :5,
    'neighbor_threshold':5,
    'maxiter'           :4,
}

cleaned_ref, mask_ref = lacosmic.remove_cosmics(image,             error=error,             **settings)
cleaned,     mask     = cusmic  .remove_cosmics(cp.asarray(image), error=cp.asarray(error), **settings)

cleaned,     mask     = cp.asnumpy(cleaned), cp.asnumpy(mask)

removed    = image   - cleaned
residual   = cleaned - baseline
difference = cleaned - cleaned_ref

## Results

The top row shares one intensity scale. **Removed signal** should resemble the
injected cosmic rays. **cusmic - lacosmic** checks implementation agreement;
**cleaned − baseline** measures reconstruction error, which need not be zero.
Residual panels have separate symmetric scales (+/-1e−12 for an all-zero residual).

In [ ]:
low, high = np.percentile(baseline, [0.5, 99.5])
image_norm  = PowerNorm(0.5, vmin=low, vmax=high)
cosmic_norm = PowerNorm(0.5, vmin=0,   vmax=np.percentile(cosmics[cosmics > 0], 99.5))
delta = max(np.abs(difference).max(), 1e-12)
limit = max(np.abs(residual).max(), 1e-12)

panels = [
    (baseline,    "before cosmic rays",               image_norm, "gray"),
    (image,       "input",                            image_norm, "gray"),
    (cleaned_ref, "cleaned: lacosmic reference",      image_norm, "gray"),
    (cleaned,     "cleaned: cusmic",                  image_norm, "gray"),
    (cosmics,     "injected cosmic rays",             cosmic_norm, "magma"),
    (removed,     "removed signal = input − cleaned", cosmic_norm, "magma"),
    (difference,  "diff = cusmic − lacosmic",         Normalize(-delta, delta), "RdBu_r"),
    (residual,    "cleaned − baseline",               Normalize(-limit, limit), "RdBu_r"),
]

fig, axes = plt.subplots(2, 4, figsize=(16, 8), layout="constrained")
for ax, (values, title, norm, cmap) in zip(axes.flat, panels):
    plot = ax.imshow(values, origin="lower", norm=norm, cmap=cmap)
    ax.set_title(title, fontsize=10)
    fig.colorbar(plot, ax=ax, shrink=0.8)
plt.show()

In [ ]:
print(f"Detected cosmic-ray pixels: {mask.sum():,}")
print(f"Mask disagreements: {np.count_nonzero(mask != mask_ref)}")
print(f"Maximum |cusmic - lacosmic|: {np.abs(difference).max():.6g}")

before = np.sqrt(np.mean((image - baseline) ** 2))
after  = np.sqrt(np.mean(residual**2))
print(f"Reconstruction RMS error: {before:.4f} → {after:.4f}")

np.testing.assert_array_equal(mask,    mask_ref)
np.testing.assert_array_equal(cleaned, cleaned_ref)

assert after < before
print("PASS: cusmic matches the reference exactly and reduces the image error.")